## Machine Learning Exercise 3: Bias and Variance

**Bias** refers to the error introduced by approximating a complex real-world problem with a simplified model, while **variance** refers to the model's sensitivity to fluctuations in the training data. A linear regression model has high bias and low variance; it makes strong assumptions about the data (linearity) but is stable across different datasets. If these strong assumptions are not correct, there will be places where it systematically overestimates or underestimates. In contrast, a decision tree model has low bias and high variance;it can capture complex patterns but is prone to overfitting, especially if deep and unpruned. This means that it can start to memorize the training data rather than capturing patterns that generalize.

In [33]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from  sklearn.metrics import mean_squared_error
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler,PolynomialFeatures
from sklearn.linear_model import LassoCV,RidgeCV

1. Fit a linear regression model to the housing data, using sqft_living to predict price. Check the mean squared error on the training data and the test data.

In [11]:
kc_sales = pd.read_csv("../data/kc_house_data.csv")
kc_sales.head()

,id,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,...,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15
0,7129300520,20141013T000000,221900.0,3,1.00,1180,5650,1.0,0,0,...,7,1180,0,1955,0,98178,47.5112,-122.257,1340,5650
1,6414100192,20141209T000000,538000.0,3,2.25,2570,7242,2.0,0,0,...,7,2170,400,1951,1991,98125,47.7210,-122.319,1690,7639
2,5631500400,20150225T000000,180000.0,2,1.00,770,10000,1.0,0,0,...,6,770,0,1933,0,98028,47.7379,-122.233,2720,8062
3,2487200875,20141209T000000,604000.0,4,3.00,1960,5000,1.0,0,0,...,7,1050,910,1965,0,98136,47.5208,-122.393,1360,5000
4,1954400510,20150218T000000,510000.0,3,2.00,1680,8080,1.0,0,0,...,8,1680,0,1987,0,98074,47.6168,-122.045,1800,7503


In [12]:
#Create Dataframe X, which contain one column, the sqft_living and a series y, which contains a target variable
predictor = ['sqft_living']
target = 'price'

X = kc_sales[predictor]
y = kc_sales[target]

#Create a traing and test set from X and y
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=321)

# Create a LinearRegression instance and fit it to training data
lg_kc_model = LinearRegression().fit(X_train, y_train)

#generate predictions on X_test and X_train
y_pred_train = lg_kc_model.predict(X_train)
y_pred_test = lg_kc_model.predict(X_test)

#calculate mean squared error on the test and training data
mse_train = mean_squared_error(y_train, y_pred_train)
mse_test = mean_squared_error(y_test, y_pred_test)

print(f'MSE_train: {mse_train}')
print(f'MSE_test: {mse_test}')


MSE_train: 69332030732.38745
MSE_test: 66079560485.167015


2. Repeat this but with a [DecisionTreeRegresor](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeRegressor.html). Again check the mean squared error on the training data and the test data. How does what you see differ from the linear regression model?

One way of avoiding overfitting is by restricting the flexibility of the model. We can do this with a decision tree by restricting the number of splits that it can perform. 

In [13]:
#Create a decisiontreeregressor instance and fit into the training data
dtree_kc_model = DecisionTreeRegressor().fit(X_train, y_train)

#generate predictions on X_test and X_train
dtree_y_pred_train = dtree_kc_model.predict(X_train)
dtree_y_pred_test = dtree_kc_model.predict(X_test)

#calculate mean squared error on the test and training data
dtree_mse_train = mean_squared_error(y_train, dtree_y_pred_train)
dtree_mse_test = mean_squared_error(y_test, dtree_y_pred_test)

print(f'dtree_MSE_train: {dtree_mse_train}')
print(f'dtree_MSE_test: {dtree_mse_test}')

print(f'test_score : {dtree_kc_model.score(X_test,y_test)}')
print(f'train_score: {dtree_kc_model.score(X_train,y_train)}')

dtree_MSE_train: 50192955708.733604
dtree_MSE_test: 73538459276.97156
test_score : 0.44425806744710084
train_score: 0.6304247784433137


There is less MSE on train data for the Decisiontreemodel compared to test data. Looks like the Decision tree regressor model performing well on train data but not on the test data(overfitting).
For linear regression model the MSE for train data and test data is almost same.

3. Fit a DecisionTreeRegressor where you restrict the max_depth to 5. Again check the mean squared error on the training data and the test data. What do you notice now?

In [14]:
#Create a decisiontreeregressor instance with max_depth 5 and fit into the training data
dtree_kc_model_depth = DecisionTreeRegressor(max_depth=5).fit(X_train, y_train)

#generate predictions on X_test and X_train
dtree_y_pred_train_depth = dtree_kc_model_depth.predict(X_train)
dtree_y_pred_test_depth = dtree_kc_model_depth.predict(X_test)

#calculate mean squared error on the test and training data
dtree_mse_train_depth = mean_squared_error(y_train, dtree_y_pred_train_depth)
dtree_mse_test_depth = mean_squared_error(y_test, dtree_y_pred_test_depth)

print(f'dtree_MSE_train: {dtree_mse_train_depth}')
print(f'dtree_MSE_test: {dtree_mse_test_depth}')

dtree_MSE_train: 60841449026.336685
dtree_MSE_test: 62926561451.64904


Restricting the max_depth to 5 for decision tree regression the MSE for test data has been reduced and train data has been in improved.There is not much difference of MSE for train and test data.
Looks like the model is performing well on test data after restricting max_depth to 5.

When working with machine learning models, we often have to balance bias and variance. This is called the [bias-variance tradeoff](https://en.wikipedia.org/wiki/Bias%E2%80%93variance_tradeoff). One method of this is through [regularization](https://www.ibm.com/think/topics/regularization), where we restrict the complexity of the model, adding some bias but reducing the variance, which can lead to a lower mean squared error on the test set.

Lasso and ridge regression do this by adding a penalty term based on the size of the coefficients. Smaller coefficients means that the model has less flexibility. The neat thing about these types of models is that they determine how to allocate the coefficients automatically as part of the model fitting process, so we can start with a large set of potential predictors and allow the model fitting to determine which ones to focus on.

For the next part of the exercise, we'll see how we can add complexity to our model but control the complexity through regularization.

4. So far, we've only been predicting based off of the square footage of living space. Fit a new linear regression model using all variables besides id, date, price, and zipcode. How well does this model perform on the test set compared to the model with just square footage of living space?

In [19]:
#Create Dataframe X, which contain all columns except 'id','date','price','zipcode', the sqft_living and a series y, which contains a target variable
predictors = kc_sales.drop(['id','date','price','zipcode'],axis='columns')
target = 'price'

X = predictors
y = kc_sales[target]

#Create a traing and test set from X and y
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=321)

# Create a LinearRegression instance and fit it to training data
multi_lg_kc_model = LinearRegression().fit(X_train, y_train)

#generate predictions on X_test and X_train
multi_y_pred_train = multi_lg_kc_model.predict(X_train)
multi_y_pred_test = multi_lg_kc_model.predict(X_test)

#calculate mean squared error on the test and training data
multi_mse_train = mean_squared_error(y_train, multi_y_pred_train)
multi_mse_test = mean_squared_error(y_test, multi_y_pred_test)

print(f'multi_MSE_train: {multi_mse_train}')
print(f'multi_MSE_test: {multi_mse_test}')

print(f'test_score : {multi_lg_kc_model.score(X_test,y_test)}')
print(f'train_score: {multi_lg_kc_model.score(X_train,y_train)}')

multi_MSE_train: 40804822162.22842
multi_MSE_test: 41757655724.75278
test_score : 0.6844306976306231
train_score: 0.6995504453115371


After adding all the features to the model the MSE for test data has been reduced and the gap between MSE of train and test data is less. Looks like this model performs well with all the features

5. Try fitting a lasso and ridge model. Becuase lasso and ridge have penalty terms based on the size of the coefficients, and the size of the coefficients depends on the scale of the variable, you'll want to scale the features first so that they are on comparable scales. Create a [Pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html) object where the first step is applying a [StandardScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html) and the second step is either a lasso or ridge model. Because these models have a hyperparameter controlling regularization strength, you'll want to use the [LassoCV](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LassoCV.html) and [RidgeCV](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.RidgeCV.html) models, which will select values for the regularization strength using cross-validation.

In [24]:
#Create Dataframe X, which contain all columns except 'id','date','price','zipcode', the sqft_living and a series y, which contains a target variable

predictors = kc_sales.drop(['id','date','price','zipcode'],axis='columns')
target = 'price'

X = predictors
y = kc_sales[target]



#Create a traing and test set from X and y
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=321)

#Create pipeline object for lasso model by applying standardscaler and LassoCV with selecting crossvalidation(cv)
lasso_pipeline = Pipeline([('scaler',StandardScaler()), 
                ('lasso_model', LassoCV(cv=5, random_state=321))])

#fit lasso model on the train data
lasso_pipeline.fit(X_train, y_train)
               
#generate predictions on X_test and X_train
lasso_pred_train = lasso_pipeline.predict(X_train)
lasso_y_pred_test = lasso_pipeline.predict(X_test)

#calculate mean squared error on the test and training data
lasso_mse_train = mean_squared_error(y_train, lasso_pred_train)
lasso_mse_test = mean_squared_error(y_test, lasso_y_pred_test)

print(f'lasso_MSE_train: {lasso_mse_train}')
print(f'lasso_MSE_test: {lasso_mse_test}')



#Create pipeline object for RidgeCV model by applying standardscaler and RidgeCV with selecting crossvalidation(cv)
ridge_pipeline = Pipeline([('scaler',StandardScaler()), 
                ('ridge_model', RidgeCV(cv=5))])

#fit lasso model on the train data
ridge_pipeline.fit(X_train, y_train)
               
#generate predictions on X_test and X_train
ridge_pred_train = ridge_pipeline.predict(X_train)
ridge_y_pred_test = ridge_pipeline.predict(X_test)

#calculate mean squared error on the test and training data
ridge_mse_train = mean_squared_error(y_train, ridge_pred_train)
ridge_mse_test = mean_squared_error(y_test, ridge_y_pred_test)

print(f'ridge_MSE_train: {ridge_mse_train}')
print(f'ridge_MSE_test: {ridge_mse_test}')


lasso_MSE_train: 40806129474.58575
lasso_MSE_test: 41770331754.19049
ridge_MSE_train: 40804846790.29033
ridge_MSE_test: 41757913805.39214


You likely didn't see much difference between the regular linear regression model and the lasso or ridge model. Let's see what happens if we add more complexity through feature interactions. We can capture more complex relationships between the predictor variables and the target variable by multiplying the predictors together before fitting the model. For example, the interaction between sqft_living and bedrooms will let the model capture if the impact of square footage depends on the number of bedrooms.

6. Add [PolynomialFeatures](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.PolynomialFeatures.html) to your pipeline after the standard scaler. Try using degree 2 features. How does this change the performance of the regular linear regression model, the lasso model, and the ridge model? 

The lasso penalty tends to cause some coeffients to zero out, so it can be viewed as a method of automatic feature selection.

In [25]:
#Create Dataframe X, which contain all columns except 'id','date','price','zipcode', the sqft_living and a series y, which contains a target variable

predictors = kc_sales.drop(['id','date','price','zipcode'],axis='columns')
target = 'price'

X = predictors
y = kc_sales[target]

#Create a traing and test set from X and y
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=321)

#Create pipeline object for lasso model by applying standardscaler , polynomialfeatures with degree of 2 and LassoCV with selecting crossvalidation(cv)
lasso_pipeline = Pipeline([('scaler',StandardScaler()), 
                           ('poly', PolynomialFeatures(2)),
                          ('lasso_model', LassoCV(cv=5, random_state=321))])

#fit lasso model on the train data
lasso_pipeline.fit(X_train, y_train)
               
#generate predictions on X_test and X_train
lasso_pred_train = lasso_pipeline.predict(X_train)
lasso_y_pred_test = lasso_pipeline.predict(X_test)

#calculate mean squared error on the test and training data
lasso_mse_train = mean_squared_error(y_train, lasso_pred_train)
lasso_mse_test = mean_squared_error(y_test, lasso_y_pred_test)

print(f'lasso_MSE_train: {lasso_mse_train}')
print(f'lasso_MSE_test: {lasso_mse_test}')


#Create pipeline object for RidgeCV model by applying standardscaler,polynomialfeatures  with degree of 2 and RidgeCV with selecting crossvalidation(cv)
ridge_pipeline = Pipeline([('scaler',StandardScaler()), 
                   ('poly', PolynomialFeatures(2)),        
                ('ridge_model', RidgeCV(cv=5))])

#fit lasso model on the train data
ridge_pipeline.fit(X_train, y_train)
               
#generate predictions on X_test and X_train
ridge_pred_train = ridge_pipeline.predict(X_train)
ridge_y_pred_test = ridge_pipeline.predict(X_test)

#calculate mean squared error on the test and training data
ridge_mse_train = mean_squared_error(y_train, ridge_pred_train)
ridge_mse_test = mean_squared_error(y_test, ridge_y_pred_test)

print(f'ridge_MSE_train: {ridge_mse_train}')
print(f'ridge_MSE_test: {ridge_mse_test}')

lasso_MSE_train: 26353345246.663406
lasso_MSE_test: 25135804989.076473
ridge_MSE_train: 25611197675.459717
ridge_MSE_test: 25216536873.773697


After adding polynomialfeatures the MSE for train and test dataset has been reduced for LassoCV and RidgeCV regression model.
It happend due to the penalty tend to cause some coefficients zero.

Why the penalties did not cause some coefficients zero when only applied StandardScaler (lassocv,ridgecv)?
why the penalty tend to cause some coefficients zero when applied polynomialfeatures?

7. Look at the set of coefficients for the lasso model. What percentage of the coefficients are zero? What are the largest non-zero coefficients?

In [37]:
#access lasso model from lasso_pipeline
lasso_model = lasso_pipeline.named_steps['lasso_model']
# checking the lasso_model coefficients
lasso_coef = lasso_model.coef_

We can see large number of lasso_model coefficients compared to number of features.
This is because polynomialfeatures(2) **creates new features** from original ones , It does not just keep the originals
Adds squared numbers and interaction numbers.
If there are **n features** degree 2 polynomial feature approximately adds **n(n+1)/2 + n**


In [46]:
#Calculate what percentage of the coefficients are zero
num_zero = np.sum(lasso_coef == 0)
total_coef = len(lasso_coef)
perc_zero = (num_zero/total_coef) * 100
print(f'Percentage of coefficients that are zero: {perc_zero:.2f}')

Percentage of coefficients that are zero: 48.54


In [ ]:
#calculate largest non-zero coefficients

8. A new hyperparameter that we have is the degree of the polynomial we're using. So that we're not overfitting to the test set, we need to use cross-validation to select this value. Set up a [GridSearchCV](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html) to try out polynomial degrees from 1 to 3 and to try LinearRegression, LassoCV, and RidgeCV models. Use 'neg_mean_squared_error' as the error_score. Which combination does it find does the best? 

** If you've reached this point, let your instructors know so that they can check in with you. **

Stretch Goals:

1. With home prices, there are some extremely large values for price, which can overly-influence the mean squared error calculation. See what happens if you optimize for mean absolute error instead. Alternatively, try using a [TransformedTargetRegressor](https://scikit-learn.org/stable/modules/generated/sklearn.compose.TransformedTargetRegressor.html) to predict the log of price instead of the raw price.

**Bonus Exercise** We've seen how a decision tree model has move flexibility, which mean higher variance compared to a linear regression model. One way of understanding variance is that variance describes how sensitive the model is to the training data. A model with high variance can produce vastly different predictions when trained on different subsets.

In this bonus exercise, you'll get to see this in action.

Generate 1000 different linear regression fits, where you only use sqft_living as the predictor variable. For each fit, choose a random sample from the full dataset of size 1000. 
Using the np.linspace function, generate a grid of 100 equally-spaced points between 500 and 3000 and generate predictions on those points. Plot the mean prediction, the 5th percentile, and the 95th percentile for the predictions from these thousand models. Repeat this for a decision tree model. Then do it for a decision tree model with a max_depth of 5.

How do these predictions differ? Which have the most variability?